### Exercise 2: Apply Pattern for Missing Values Handling
#### Objective:
• Learn the Strategy Pattern - one interface, many interchangeable behaviors.

• Practice inheritance and polymorphism.

#### Concept:
We often need different strategies to handle missing data (drop, fill mean, fill mode, etc.). The pattern allows flexible switching between methods without changing the main code.
#### Instructions:
1. Create an abstract class MissingValueStrategy with a method handle(df).
2. Create subclasses:

o DropMissing

o FillMean

o FillMode

3. Create a DataCleaner class that applies the chosen strategy.

In [4]:
# import tools and locast CSV
from pathlib import Path
from abc import ABC, abstractmethod

import pandas as pd
from IPython.display import display

In [5]:
INPUT_PATH = Path("sample_data (2).csv")

df = pd.read_csv(INPUT_PATH)
original_df = df.copy(deep=True)

display(df)

,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


In [6]:
print("Shape:", df.shape)

Shape: (7, 7)


In [7]:
display(df.isna().sum())

id           0
name         0
age          3
height_cm    1
weight_kg    1
city         0
score        2
dtype: int64

In [8]:
# define common strategy intreface
class MissingValueStrategy(ABC):
    """Common interface for missing-value handling strategies."""

    @abstractmethod
    def handle(self, df):
        """Return a cleaned DataFrame without modifying the input."""
        pass

In [9]:
# Implement DeopMissingStrategy
class DropMissing(MissingValueStrategy):
    def handle(self, df):
        return df.dropna().copy()

In [10]:
# Implement Fillmean
class FillMean(MissingValueStrategy):
    def handle(self, df):
        result = df.copy()
        numeric_columns = result.select_dtypes(include="number").columns

        for column in numeric_columns:
            mean_value = result[column].mean()
            result[column] = result[column].fillna(mean_value)

        return result

In [11]:
# Implement FillMode
class FillMode(MissingValueStrategy):
    def handle(self, df):
        result = df.copy()

        for column in result.columns:
            modes = result[column].mode()
            if not modes.empty:
                result[column] = result[column].fillna(modes.iloc[0])

        return result

In [12]:
#Create dataCleaner
class DataCleaner:
    def __init__(self, strategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.handle(df)


cleaner = DataCleaner(DropMissing())
cleaned_df = cleaner.clean(df)
display(cleaned_df)

,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
4,5,Eva,27.0,160.0,55.0,New York,88.0


In [13]:
class DataCleaner:
    def __init__(self, strategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.handle(df)


cleaner = DataCleaner(FillMean())
cleaned_df = cleaner.clean(df)
display(cleaned_df)

,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.00,165.0,68.0,New York,85.0
1,2,Bob,30.25,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.00,168.0,72.0,Chicago,86.0
3,4,David,30.25,167.0,80.0,Houston,75.0
4,5,Eva,27.00,160.0,55.0,New York,88.0
5,6,Frank,30.00,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.25,162.0,60.0,Chicago,92.0


In [14]:
class DataCleaner:
    def __init__(self, strategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.handle(df)


cleaner = DataCleaner(FillMode())
cleaned_df = cleaner.clean(df)
display(cleaned_df)

,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,27.0,172.0,55.0,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,75.0
3,4,David,27.0,160.0,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,75.0
6,7,Grace,27.0,162.0,60.0,Chicago,92.0


In [15]:
# Switech strategy on the same cleaner
cleaner = DataCleaner(FillMean())
mean_result = cleaner.clean(df)

cleaner.strategy = DropMissing()
drop_result = cleaner.clean(df)

cleaner.strategy = FillMode()
mode_result = cleaner.clean(df)

display(pd.DataFrame({
    "name": df["name"],
    "original_age": df["age"],
    "mean_filled_age": mean_result["age"],
    "mode_filled_age": mode_result["age"],
}))

,name,original_age,mean_filled_age,mode_filled_age
0,Alice,29.0,29.00,29.0
1,Bob,NaN,30.25,27.0
2,Charlie,35.0,35.00,35.0
3,David,NaN,30.25,27.0
4,Eva,27.0,27.00,27.0
5,Frank,30.0,30.00,30.0
6,Grace,NaN,30.25,27.0
